# 1 — Data Exploration & Preprocessing

This notebook loads the corporate credit rating dataset, explores its structure, and runs the preprocessing pipeline that produces the train/test split used by all subsequent notebooks.

The preprocessing pipeline follows these steps:

1. **Load** the raw dataset
2. **Collapse** notched ratings (e.g. AAA-, AAA, AAA+ → AAA) to get 10 broad classes
3. **Winsorize** features at the 1st/99th percentile to reduce outlier influence
4. **Encode** the 10 rating classes as integers (D=0, C=1, ... AAA=9)
5. **Split** into train/test using a company-aware group split to prevent data leakage
6. **Save** the split to `outputs/splits.pkl`

The core logic for each step is in `src/data.py`. This notebook calls those functions and visualises the data at each stage.

## Setup

The cell below clones the repository from GitHub (if running in Colab) and makes the `src` package available for import. If running locally, it detects that `src/` already exists and skips the clone.

In [ ]:
import os, sys

if not os.path.isdir("src"):
    !git clone https://github.com/andreasz24/Thesis-Artifact.git
    %cd Thesis-Artifact
    !pip install -r requirements.txt --quiet

sys.path.insert(0, os.getcwd())
print("Ready! Working directory:", os.getcwd())

## Loading the raw dataset

The dataset contains 7,805 observations from 7 credit rating agencies. Each row represents one rating event — a company being rated by a specific agency at a point in time. The dataset includes 16 financial ratios as explanatory variables and the credit rating as the target variable.

Source: [Kaggle — Corporate Credit Rating with Financial Ratios](https://www.kaggle.com/datasets/kirtandelwadia/corporate-credit-rating-with-financial-ratios)

In [ ]:
import pandas as pd
from src import data, config

df = data.load_raw()
print(f'Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

### Column overview

The 16 financial ratio features cover five domains of financial health identified in the credit literature (Horrigan, 1966; Altman, 1968), plus cash flow:

- **Liquidity:** Current Ratio
- **Leverage:** Long-term Debt / Capital, Debt/Equity Ratio
- **Profitability:** Gross Margin, Operating Margin, EBIT Margin, EBITDA Margin, Pre-Tax Profit Margin, Net Profit Margin, ROA, ROE, ROI, Return on Tangible Equity
- **Activity:** Asset Turnover
- **Cash Flow:** Operating Cash Flow Per Share, Free Cash Flow Per Share

In [ ]:
# The 16 features used as model inputs
print('Feature columns:')
for i, col in enumerate(config.FEATURE_COLS, 1):
    print(f'  {i:2d}. {col}')

print(f'\nNumber of unique companies: {df[config.GROUP_COL].nunique()}')
print(f'Number of rating agencies: {df["Rating Agency"].nunique()}')

## Target variable: the rating distribution

Before preprocessing, the rating column contains 23 distinct classes. Each broad rating grade (e.g. AAA) has up to three notched variants: AAA-, AAA, and AAA+. The cell below shows how observations are distributed across these 23 classes.

In [ ]:
print(f'Number of unique ratings before collapsing: {df[config.RATING_COL].nunique()}')
df[config.RATING_COL].value_counts()

## Collapsing notched ratings

We strip the `+` and `-` suffixes from each rating string so that AAA-, AAA, and AAA+ all become AAA. This reduces the 23 classes to 10 broad rating categories.

This is done because the notched distinctions are too fine-grained for the models to learn reliably from this dataset size, and because the thesis research question focuses on broad rating class prediction.

In [ ]:
collapsed = data.collapse_ratings(df)
print(f'Number of unique ratings after collapsing: {collapsed[config.RATING_COL].nunique()}')
print(f'\nRating distribution (ordered from D to AAA):')
collapsed[config.RATING_COL].value_counts().reindex(config.RATING_ORDER)

### The class imbalance problem

The distribution above shows a significant class imbalance. The investment-grade categories (BBB, A) and the upper speculative category (BB) dominate the dataset, while the extreme classes at both ends of the scale (C, CC, D, AAA) have very few observations.

This imbalance is the single most important factor shaping model behaviour in this study. Models trained on this distribution will inevitably favour the common classes and struggle with the rare ones — which is exactly what we observe in the results.

Techniques like SMOTE (Synthetic Minority Oversampling) were considered but not applied, since the goal of this study is to **compare** model performance under identical conditions, not to optimise any individual model.

## Winsorizing outliers

Financial ratios can contain extreme outliers (e.g. a company with a debt/equity ratio of 10,000% due to near-zero equity). These outliers can distort model training.

Winsorization clips each feature at its 1st and 99th percentile — values below the 1st percentile are set to the 1st percentile value, and values above the 99th are set to the 99th. This preserves the overall distribution while reducing the influence of extreme values.

In [ ]:
# Show an example: Debt/Equity Ratio before and after winsorization
example_col = 'Debt/Equity Ratio'
print(f'{example_col} BEFORE winsorization:')
print(f'  Min: {collapsed[example_col].min():.2f}  Max: {collapsed[example_col].max():.2f}')

winsorized = data.winsorize(collapsed)
print(f'\n{example_col} AFTER winsorization:')
print(f'  Min: {winsorized[example_col].min():.2f}  Max: {winsorized[example_col].max():.2f}')

## Encoding ratings as integers

The 10 broad rating classes are encoded as consecutive integers in ascending order of creditworthiness:

| Rating | D | C | CC | CCC | B | BB | BBB | A | AA | AAA |
|--------|---|---|----|-----|---|----|-----|---|----|-----|
| Code   | 0 | 1 | 2  | 3   | 4 | 5  | 6   | 7 | 8  | 9   |

This integer encoding allows the models to treat the target as a standard multiclass classification problem. It also means that the size of a prediction error can be interpreted ordinally — predicting B (4) for a true BB (5) is a one-step error, while predicting CCC (3) would be a two-step error.

## Train/test split: preventing data leakage

A critical methodological decision in this study is how the data is split into training and test sets.

The problem: many companies appear multiple times in the dataset (rated repeatedly, or by multiple agencies). A standard random split would place some observations of the same company in both train and test. The model would then effectively memorise a company's rating during training and recognise it during testing — this is called **data leakage**, and it inflates performance metrics.

The solution: a **group-based split** using `GroupShuffleSplit` with the company name as the grouping variable. This ensures that ALL observations of a given company land exclusively in either the training set or the test set, never both.

The function `data.build_splits()` runs the full pipeline (collapse → winsorize → encode → group split) and saves the result.

In [ ]:
X_train, X_test, y_train, y_test = data.build_splits()

## Split summary

The split is now saved to `outputs/splits.pkl`. The other notebooks (2–4) load this file at the start, guaranteeing that all three models are trained and evaluated on exactly the same data.

In [ ]:
print(f'Training set: {X_train.shape[0]} rows, {X_train.shape[1]} features')
print(f'Test set:     {X_test.shape[0]} rows, {X_test.shape[1]} features')
print(f'\nClass distribution in test set:')
for code in sorted(y_test.unique()):
    label = config.RATING_LABELS[int(code)]
    count = (y_test == code).sum()
    print(f'  {label:>3s} (code {int(code)}): {count:4d} observations')